# Pre-T：Program Identity → Enrichment → Usage → Direction → Integration

本 Notebook 使用新版 Layer0–3 接口；同一张基因表从身份标注沿用到最终组内驱动标注。
富集和 usage 独立，方向标注随后运行。此文件是最终维护入口；历史实现可从 Git 历史查看。

运行前安装项目依赖，选择正确 Python 内核并把工作目录设为仓库根目录。
真实数据没有随仓库提供，下面的主流程需要 cNMF/H5AD/注释输入；wPSI 缺失时明确跳过方向分析。
完整输出写入 `analysis_results/`，不覆盖旧的 `outputs/`。

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display
from depa_downstream import (
    load_cnmf_outputs, build_program_gene_tables, run_functional_enrichment,
    compute_contrast_association, summarize_program_associations,
    compute_population_direction, annotate_within_celltype,
    plot_enrichment_bubble, plot_association_heatmap, plot_program_umap_grid,
    plot_population_direction, plot_within_celltype_concordance,
)
from depa_downstream.pipeline import load_analysis_config, assemble_analysis_results, export_analysis

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'pyproject.toml').is_file():
    raise RuntimeError('请先把 Notebook 工作目录切换到 DEPA_downstream 仓库根目录')

## 配置与数据加载
只加载一次配置；后续单元不再重新读 YAML，也不再手动指定 program→celltype。

In [ ]:
# 只改数据目录；原始 YAML 的 Mac 绝对路径不会被覆盖。
DATA_ROOT = PROJECT_ROOT / 'data' / 'pre_t'
CONFIG = load_analysis_config(
    PROJECT_ROOT / 'configs' / 'pre_t.yaml', root_dir=PROJECT_ROOT,
    output_root=PROJECT_ROOT / 'analysis_results' / 'pre_t',
    path_overrides={
        'cnmf_dir': str(DATA_ROOT / 'cNMF_output'),
        'h5ad_path': str(DATA_ROOT / 'cNMF_output' / 'PreT_DEAPA_counts.h5ad'),
        'feature_anno_path': str(DATA_ROOT / 'cNMF_input' / 'cnmf_feature_anno.csv'),
        'wpsi_matrix_path': str(DATA_ROOT / 'PSI_results' / 'movAPA_wPSI_matrix_df.rds'),
        'gene_symbol_map_path': str(PROJECT_ROOT / 'configs' / 'pre_t_gene_symbol_map.csv'),
    },
)

OUTPUT_ROOT = Path(CONFIG['output_root'])
data = load_cnmf_outputs(CONFIG)

## Layer 0 · Program Identity
PAC→gene 按最大权重和最大 Z 聚合。`is_core` 使用 ≥ 阈值；低 wPSI 检出率只标注，不删除核心基因。后续各层只读取这些标志。

In [ ]:
gene_tables = build_program_gene_tables(data, CONFIG)
identity_summary = pd.DataFrame({p: {'n_genes': len(t), 'n_core': int(t.is_core.sum()), 'n_specific': int(t.is_specific.sum()), 'n_reliable': int(t.is_wpsi_reliable.fillna(False).sum())} for p,t in gene_tables.items()}).T
display(identity_summary)

## Layer 1 · Functional Enrichment
分别对核心基因和专属 marker 查询；背景为当前 PAC 基因全集的可用 symbol，使用 Speedrichr 自定义背景接口。
失败、结果为空和基因太少分别标记；无显著富集不自动判定 cNMF 失败。此时不使用方向性信息。

In [ ]:
enrichment = run_functional_enrichment(gene_tables, CONFIG, on_error='record')
enrichment_status = pd.DataFrame([dict(program=p, gene_set=k, **{a:v for a,v in t.attrs.items() if a not in ('program','gene_set')}) for p,sets in enrichment.items() for k,t in sets.items()])
display(enrichment_status)
for kind in ('core', 'specific'):
    plot_enrichment_bubble({p: sets[kind] for p,sets in enrichment.items()}, str(OUTPUT_ROOT / 'layer1'), filename=f'layer1_enrichment_{kind}.png')

## Layer 2 · Usage Association
仅使用 cells×program 的 usage 与 metadata。最佳细胞类型按 |AUC−0.5| 选取；同分按类型名排序，低 AUC 表示耗竭关联而非富集。

In [ ]:
assoc_results = compute_contrast_association({'usage': data['usage'], 'meta': data['meta']}, CONFIG)
association_summary = summarize_program_associations(assoc_results, CONFIG, programs=list(gene_tables))
display(association_summary)
plot_association_heatmap(assoc_results, CONFIG, str(OUTPUT_ROOT / 'layer2'))
plot_program_umap_grid(data, CONFIG, str(OUTPUT_ROOT / 'layer2'))

## Layer 3a · 群体方向检验
读取 Layer0 缓存的可靠基因中位数；默认背景是可靠 PAC universe 中剔除当前前景的基因，与旧版 MWU 的不重叠比较一致。
`expected_direction` 明确表示 usage **升高**时预期的 wPSI 增减，不等同于疾病组 B−A 的符号。
方向是群体前景相对背景的描述，`is_significant` 独立记录 BH 显著性。

In [ ]:
population_direction = compute_population_direction(gene_tables, CONFIG)
display(population_direction)
plot_population_direction(population_direction, str(OUTPUT_ROOT / 'layer3'), filename='layer3_population_direction.png')

## Layer 3b · 自动选择细胞类型并写回基因表
逐个 program 使用 Layer2 选出的细胞类型，按疾病两组检验全部核心基因。
`within_celltype_mean_diff` 固定为 B−A；一致性比较会乘以该细胞类型内 usage(B−A) 的符号，避免把 usage 下降的 program 判反。
细胞级 MWU 用于探索性标注，不能替代 donor/sample 级独立重复验证；未检验项与不确定项分开保留。

In [ ]:
annotate_within_celltype(gene_tables, data, CONFIG, association_summary, population_direction)
concordance_summary = pd.DataFrame({p: t.loc[t.is_core, 'concordance'].value_counts() for p,t in gene_tables.items()}).fillna(0).astype(int).T
display(concordance_summary)
for program in gene_tables:
    plot_within_celltype_concordance(gene_tables, program, str(OUTPUT_ROOT / 'layer3'))

## 收尾整合 · Enrichment × Concordance
不重新富集、不重新筛选核心基因。对每个 term 追加 concordant/discordant/inconclusive 的数量、symbol 和 gene ID；未检验和未匹配的命中单独记录。

In [ ]:
results = assemble_analysis_results(gene_tables, enrichment, assoc_results, association_summary, population_direction, CONFIG)
integrated_enrichment = results['integrated_enrichment']
export_analysis(results, OUTPUT_ROOT)
print('分析执行状态:', results['manifest']['status'])
print('结果目录:', OUTPUT_ROOT)
report_terms = pd.concat([t.assign(program=p, gene_set=k) for p,sets in integrated_enrichment.items() for k,t in sets.items()], ignore_index=True)
if not report_terms.empty:
    display(report_terms.sort_values('padj').head(30))

## P02 专项报告 · 基因特异的 3′UTR 长短偏好
复用 Layer0 核心基因、Layer2 自动选定的细胞类型和 Layer3 方向结果。所有新增文件保存至原 `layer3`，以 `P02_` 开头。
主图默认使用 P02 内 BH q<0.05；同时输出原始 p 分类，与前面的探索性 concordance 区分。
先核对上游 wPSI 增大是否代表远端/较长 3′UTR，以及未检出值是否保留为 NaN。
蓝色=疾病组偏短，橙色=偏长；不把富集命中数当作子集富集检验。

In [ ]:
from p02_report import generate_p02_report

p02_report = generate_p02_report(
    data, CONFIG, gene_tables, association_summary, population_direction,
    output_dir=OUTPUT_ROOT / 'layer3',
    enrichment=enrichment,
    significance='padj',  # 报告主图用 BH q；'p' 可复现原始 p 的探索性分类
    key_genes=['CENPM', 'NAP1L1', 'TPX2', 'MPC1', 'SOX4', 'STMN1',
               'PTP4A2', 'PCLAF', 'HNRNPA1', 'HNRNPA0', 'CACYBP', 'PSIP1'],
)
display(p02_report['genes'][[
    'symbol', 'spectra_z', 'is_specific', 'within_celltype_mean_a',
    'within_celltype_mean_b', 'within_celltype_mean_diff',
    'within_celltype_p', 'within_celltype_padj', 'report_length_preference',
    'sample_mean_diff', 'sample_padj', 'sample_status',
]])
print('P02 报告目录:', p02_report['output_dir'])
print('原始 p 分类:', p02_report['summary']['raw_p_counts'])
print('BH q 分类:', p02_report['summary']['bh_counts'])

from IPython.display import Image
for name, paths in p02_report['figures'].items():
    display(Image(filename=next(p for p in paths if p.endswith('.png')), width=1000))
